# Lesson 2: RAG Triad of metrics (RAG 指标三元组)

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [10]:
import importlib
import utils
importlib.reload(utils)

import os
import openai

In [11]:
from trulens.core import Tru 

# 创建 Tru 类的实例对象
tru = Tru()
# 重置数据库，删除所有之前存储的评估数据
tru.reset_database()

Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


In [46]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()

In [47]:
from llama_index.core import Document

document = Document(text="\n\n".\
                    join([doc.text for doc in documents]))
print(document)

Doc ID: 3dac8952-b97f-4775-9648-a83c985e9a45
Text: PAGE 1 Founder, DeepLearning.AI Collected Insights from Andrew
Ng How to  Build Your Career in AI A Simple Guide   PAGE 2 "AI is the
new  electricity. It will  transform and improve  all areas of human
life." Andrew Ng  PAGE 3 Table of  Contents Introduction: Coding AI is
the New Literacy. Chapter 1: Three Steps to Career Growth. Chapter 2:
Lear...


In [48]:
from utils import build_sentence_window_index

from llama_index.core import Settings
from llama_index.llms.openai_like import OpenAILike



# OpenAILike: LlamaIndex 中用于连接与 OpenAI API 兼容的服务的类
# 这里用于连接阿里云的通义千问 (DashScope) 服务
#    配置了 API Key、基础 URL (api_base) 和模型 (qwen-max)。
#    设置了较低的 temperature=0.1，以获得更确定性的回答。
#    设置了较大的 context_window=128000。
llm = OpenAILike(
    api_key=utils.get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=False,
)

model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
# 使用自定义函数构建或加载 Sentence Window Index（句子窗口索引）。
# 这是 LlamaIndex 中一种优化检索的方法：检索小块（句子），但提供大块上下文（窗口）给 LLM。
sentence_index = build_sentence_window_index(
    document,
    llm,
    embed_model=model_real_path,
    save_dir="sentence_index"
)

In [49]:
from utils import get_sentence_window_query_engine

# 获取 Sentence Window 查询引擎。该引擎会自动执行 "检索句子+回填窗口上下文" 的逻辑。
sentence_window_engine = \
get_sentence_window_query_engine(sentence_index)

In [50]:
output = sentence_window_engine.query(
    "你如何创建你的人工智能投资组合?")
output.response

'创建人工智能投资组合时，你可以遵循以下步骤：\n\n1. **确定业务问题**：首先，找到一个领域专家并询问他们最希望哪些方面能够改进以及为什么这些问题尚未得到解决。这有助于你识别出真正需要解决的问题，而不是仅仅关注技术本身。\n\n2. **头脑风暴AI解决方案**：理解了具体问题之后，开始思考可能的AI解决方案。不要急于实施第一个想到的想法，而是花时间去探索不同的可能性，这样可能会发现更优解且实现起来并不一定更加复杂。\n\n通过这种方式，你可以展示自己解决问题的能力和技术进步的过程，这对于构建一个有吸引力的人工智能项目集非常关键。'

## Feedback functions (反馈函数)

In [51]:
import nest_asyncio
# ----------------------------------------------------------------------
# 异步环境配置
# ----------------------------------------------------------------------
# nest_asyncio.apply()：用于解决在Jupyter/Colab环境中运行异步代码时，
#                       事件循环可能已经运行的问题。确保TruLens的异步评估能正常工作。
nest_asyncio.apply()

In [52]:
import os
# from trulens_eval import OpenAI as fOpenAI
# provider = fOpenAI()
from trulens.providers.litellm import LiteLLM# 导入 LiteLLM 提供者

dashscope_key = os.environ.get("DASHSCOPE_API_KEY")

# 2. 核心补丁：强行注入 OpenAI 的环境变量
# 这会引导 LiteLLM 走它最熟悉的 OpenAI 稳定通道
os.environ["OPENAI_API_KEY"] = dashscope_key
os.environ["OPENAI_API_BASE"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"

# 3. 实例化 Provider
# liteLLM_provider：使用 LiteLLM 封装的提供者，将大模型服务（如 DashScope/Qwen-Max）
#                 接入 TruLens，用于执行评估任务（如相关性判断、事实一致性检查）。
provider = LiteLLM(model_engine="openai/qwen-max")

print("✅ Provider 实例化成功，已通过 OpenAI 兼容模式连接 Qwen-Max")

✅ Provider 实例化成功，已通过 OpenAI 兼容模式连接 Qwen-Max


### 1. Answer Relevance (答案相关性)

In [53]:
from trulens.core import Feedback 
# ----------------------------------------------------------------------
# TruLens 评估指标定义 (Feedback Functions)
# ----------------------------------------------------------------------
# 回答相关性：衡量的是 Input (问题) 和 Output (回答) 之间的相关性
f_qa_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()  # 评估的第一个输入是用户问题 (Input)
    .on_output() # 评估的第二个输入是 LLM 的答案 (Output)
)


### 2. Context Relevance (上下文相关性)

In [54]:
from trulens.apps.llamaindex import TruLlama
# 定义一个数据选择器（selector），用于指定 TruLens 在评估过程中应捕获和记录 RAG 流程中的哪个特定部分作为上下文。
context_selection = TruLlama.select_source_nodes().node.text

In [55]:
import numpy as np

# --- 第一种 Context Relevance 评估：基础相关性 ---
f_qs_relevance = (
    Feedback(provider.relevance,
             name="Context Relevance")
    .on_input() # 指定第一个输入：用户查询（Question）。
    .on_context(collect_list=True) # 指定第二个输入：检索到的上下文（Source Nodes）。
                                    # collect_list=True 确保将检索到的所有上下文块作为一个列表收集。
    .aggregate(np.mean) # 聚合函数：对所有检索到的上下文块的得分求平均值，作为最终的“Context Relevance”分数。
)

In [56]:
import numpy as np

# --- 第二种 Context Relevance 评估：带COT理由的相关性 ---
f_qs_relevance = (
    Feedback(
        # 使用带思维链（Chain-of-Thought, COT）理由的问题-源相关性评估器。
        # 相比 qs_relevance，这个评估器会要求LLM先给出推理过程，再给出分数，增加了可解释性。
        provider.relevance_with_cot_reasons,
        name="Context Relevance")
    .on_input() # 指定第一个输入：用户查询（Question）
    .on_context(collect_list=True) # 指定第二个输入：检索到的上下文（Source Nodes），以列表形式收集。
    .aggregate(np.mean) # 聚合函数：对所有上下文块的COT相关性得分求平均值。
)

### 3. Groundedness (基础性)

In [57]:
f_groundedness = (
    Feedback(
        # 使用 provider（通常是一个LLM）提供的、带有思维链（COT）推理的事实基础性评估方法。
        # 这个函数会判断答案中的每个陈述句是否在上下文中被支持。
        provider.groundedness_measure_with_cot_reasons,
        name="Groundedness"
    )
    .on_context(collect_list=True)
    .on_output()
    .aggregate(np.mean)
)

## Evaluation of the RAG application (RAG 应用的评估)

In [ ]:
from trulens.apps.llamaindex import TruLlama
from trulens.core import FeedbackMode

tru_recorder = TruLlama(
    sentence_window_engine,
    app_id="App_1",
    feedbacks=[
        f_qa_relevance,
        f_qs_relevance,
        f_groundedness
    ]
)

[nltk_data] Error loading punkt_tab: Remote end closed connection
[nltk_data]     without response
[nltk_data] Error loading punkt_tab: Remote end closed connection
[nltk_data]     without response


In [63]:
eval_questions = []
with open('eval_questions.txt', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [60]:
eval_questions

['What are the keys to building a career in AI?',
 'How can teamwork contribute to success in AI?',
 'What is the importance of networking in AI?',
 'What are some good habits to develop for a successful career?',
 'How can altruism be beneficial in building a career?',
 'What is imposter syndrome and how does it relate to AI?',
 'Who are some accomplished individuals who have experienced imposter syndrome?',
 'What is the first step to becoming good at AI?',
 'What are some common challenges in AI?',
 'Is it normal to find parts of AI challenging?']

In [61]:
eval_questions.append("我怎样才能在AI中获得成功?")

In [62]:
eval_questions

['What are the keys to building a career in AI?',
 'How can teamwork contribute to success in AI?',
 'What is the importance of networking in AI?',
 'What are some good habits to develop for a successful career?',
 'How can altruism be beneficial in building a career?',
 'What is imposter syndrome and how does it relate to AI?',
 'Who are some accomplished individuals who have experienced imposter syndrome?',
 'What is the first step to becoming good at AI?',
 'What are some common challenges in AI?',
 'Is it normal to find parts of AI challenging?',
 '我怎样才能在AI中获得成功?']

In [65]:
for question in eval_questions:
    with tru_recorder as recording:
        sentence_window_engine.query(question)

In [66]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
records.head()

app_id app_name app_version  \
0  app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e    App_1        base   
1  app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e    App_1        base   
2  app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e    App_1        base   
3  app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e    App_1        base   
4  app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e    App_1        base   

                                                                                              app_json  \
0  {'app_name': 'App_1', 'app_version': 'base', 'app_id': 'app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e'}   
1  {'app_name': 'App_1', 'app_version': 'base', 'app_id': 'app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e'}   
2  {'app_name': 'App_1', 'app_version': 'base', 'app_id': 'app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e'}   
3  {'app_name': 'App_1', 'app_version': 'base', 'app_id': 'app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e'}   
4  {'app_name': 'App_1', 'app_version': 'base', 'app_id': 'app_hash_9a8bbe93b76fc0a0eca1a6fc7bd3027e'}   

   type                             record_id input_id                 input  \
0  SPAN  181f1ce8-1d55-4a75-a04e-7715aaab549f           在人工智能领域建立职业生涯的关键是什么？   
1  SPAN  9b711bcb-e591-425f-9ede-b1affd0e378a              团队合作如何有助于人工智能的成功？   
2  SPAN  d4e8a5d7-0ba6-4a80-ab48-c432322e30c3               网络在人工智能中的重要性是什么？   
3  SPAN  5a3ce7fb-7522-4fd4-8fab-ba980518fbca                 要想事业成功，有哪些好习惯？   
4  SPAN  258a0e1d-b706-494e-bb0e-2b39118bdf2d                利他主义如何有益于事业的发展？   

                                                                                                                                                                                                                                                                                                                                                                                              output  \
0                                                                                                                                                                                                                                                                    在人工智能领域建立职业生涯的关键包括学习基础技术技能、参与项目以及找到工作。同时，构建一个支持性的社区也非常重要，这个社区可以提供帮助，并且你也可以努力去帮助他人。此外，拓展人脉网络也很关键，因为认识的人不仅可以提供宝贵的信息，还可以推荐你给潜在的雇主。   
1                                                                                                                                                                                                   团队合作在人工智能领域的成功中起着至关重要的作用。当团队成员能够有效地协作时，他们可以汇集各自的专业知识和技能，共同解决复杂的问题。这种合作不仅促进了创新思维和技术进步，还帮助团队更快速地克服挑战。通过共享资源、相互学习和支持，团队能够提高工作效率，确保项目的顺利进行。此外，良好的团队合作还能营造一个积极的工作环境，鼓励每位成员发挥出最佳状态，这对于推动人工智能技术的发展至关重要。   
2                                                                                                                                                                                                                                                                在网络和人工智能领域，建立良好的人际关系网络非常重要。通过与他人交流，你可以获得宝贵的信息，并且这些人脉关系可能会推荐你给潜在的雇主。此外，参加像Pie & AI这样的聚会可以帮助你扩展人脉。当你在职业道路上取得进展时，也不要忘了回馈社区，帮助那些正在起步的人。   
3  要想在事业上取得成功，可以培养以下几个好习惯：\n\n1. **持续学习与实践**：从简单的项目开始，逐步挑战更复杂的任务。这不仅有助于积累经验，还能展示你的成长轨迹，对求职非常有帮助。\n\n2. **有效沟通**：无论是启动新项目还是完成现有工作，都需要能够清晰地表达自己的想法和成果。良好的沟通能力可以帮助你获得同事、导师以及上级的支持，并且让他们看到你工作的价值。\n\n3. **快速迭代**：面对不确定的情况时，先快速构建一个完整的系统，然后根据反馈不断调整优化。这种方法可以在早期阶段就发现问题并及时修正方向。\n\n4. **审慎决策**：对于那些一旦决定就难以更改的重大投资或选择，应该花更多时间进行前期研究和规划，确保所选路径是正确的。在高度自信之前不要轻易做出承诺。\n\n通过这些习惯的养成，可以提高工作效率和个人影响力，从而促进职业生涯的发展。   
4                                                                                                                                        利他主义可以通过多种方式有益于事业的发展。首先，通过帮助他人解决问题，你可以建立一个展示你技能和成长的项目组合，这在求职时会非常有帮助。其次，良好的沟通能力是至关重要的，当你能够清晰地解释你的想法和成果时，更容易获得同事、导师和管理层的支持与资源，从而有机会参与更大规模的项目。此外，通过与不同领域的专家合作，了解他们面临的挑战，并尝试用人工智能等技术提供解决方案，可以让你在不熟悉的行业中找到有意义的工作机会。总之，采取一种以解决实际业务问题为中心的方法，不仅有助于个人职业发展，也能为社会创造价值。   

  tags  ... Answer Relevance feedback cost in USD Answer Relevance direction  \
0       ...                                   0.0                       

In [67]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

,input,output,Answer Relevance,Context Relevance,Groundedness
0,在人工智能领域建立职业生涯的关键是什么？,在人工智能领域建立职业生涯的关键包括学习基础技术技能、参与项目以及找到工作。同时，构建一个支持性的社区也非常重要，这个社区可以提供帮助，并且你也可以努力去帮助他人。此外，拓展人脉网络也很关键，因为认识的人不仅可以提供宝贵的信息，还可以推荐你给潜在的雇主。,NaN,NaN,NaN
1,团队合作如何有助于人工智能的成功？,团队合作在人工智能领域的成功中起着至关重要的作用。当团队成员能够有效地协作时，他们可以汇集各自的专业知识和技能，共同解决复杂的问题。这种合作不仅促进了创新思维和技术进步，还帮助团队更快速地克服挑战。通过共享资源、相互学习和支持，团队能够提高工作效率，确保项目的顺利进行。此外，良好的团队合作还能营造一个积极的工作环境，鼓励每位成员发挥出最佳状态，这对于推动人工智能技术的发展至关重要。,1.0,0.0,0.000000
2,网络在人工智能中的重要性是什么？,在网络和人工智能领域，建立良好的人际关系网络非常重要。通过与他人交流，你可以获得宝贵的信息，并且这些人脉关系可能会推荐你给潜在的雇主。此外，参加像Pie & AI这样的聚会可以帮助你扩展人脉。当你在职业道路上取得进展时，也不要忘了回馈社区，帮助那些正在起步的人。,0.0,0.0,1.000000
3,要想事业成功，有哪些好习惯？,要想在事业上取得成功，可以培养以下几个好习惯：\n\n1. **持续学习与实践**：从简单的项目开始，逐步挑战更复杂的任务。这不仅有助于积累经验，还能展示你的成长轨迹，对求职非常有帮助。\n\n2. **有效沟通**：无论是启动新项目还是完成现有工作，都需要能够清晰地表达自己的想法和成果。良好的沟通能力可以帮助你获得同事、导师以及上级的支持，并且让他们看到你工作的价值。\n\n3. **快速迭代**：面对不确定的情况时，先快速构建一个完整的系统，然后根据反馈不断调整优化。这种方法可以在早期阶段就发现问题并及时修正方向。\n\n4. **审慎决策**：对于那些一旦决定就难以更改的重大投资或选择，应该花更多时间进行前期研究和规划，确保所选路径是正确的。在高度自信之前不要轻易做出承诺。\n\n通过这些习惯的养成，可以提高工作效率和个人影响力，从而促进职业生涯的发展。,1.0,0.0,0.933333
4,利他主义如何有益于事业的发展？,利他主义可以通过多种方式有益于事业的发展。首先，通过帮助他人解决问题，你可以建立一个展示你技能和成长的项目组合，这在求职时会非常有帮助。其次，良好的沟通能力是至关重要的，当你能够清晰地解释你的想法和成果时，更容易获得同事、导师和管理层的支持与资源，从而有机会参与更大规模的项目。此外，通过与不同领域的专家合作，了解他们面临的挑战，并尝试用人工智能等技术提供解决方案，可以让你在不熟悉的行业中找到有意义的工作机会。总之，采取一种以解决实际业务问题为中心的方法，不仅有助于个人职业发展，也能为社会创造价值。,NaN,NaN,0.777778
5,什么是冒名顶替综合症？它与人工智能有什么关系？,提供的信息中没有提到冒名顶替综合症或其与人工智能的关系。根据你所提供的内容，我无法直接回答这个问题。不过，通常来说，冒名顶替综合症是指一个人即使在取得成功后仍然持续感到自己不够格，并担心被揭穿的一种心理状态。至于它与人工智能的关系，这可能涉及到人们在使用或开发AI技术时感受到的不安全感或自我怀疑，但这部分内容并没有出现在给定的信息里。,NaN,NaN,NaN
6,有哪些成功人士经历过冒名顶替综合症？,据估计，大约70%的人在某个时刻会经历某种形式的冒名顶替综合症，这包括许多成功人士。即使在人工智能领域取得了一定成就的人，有时也会怀疑自己是否真的属于这个领域，担心自己被看作是骗子。这种感觉并不罕见，也不应该成为阻碍个人成长和发展的因素。,NaN,NaN,NaN
7,精通人工智能的第一步是什么？,精通人工智能的第一步是学习技术技能，为有前景的人工智能职业生涯打下基础。这包括掌握编程等基本技能，因为编码AI被视为新的读写能力。此外，建立你的人脉网络也很重要，可以通过参加像Pie & AI这样的聚会来实现。同时，学习过程中保持礼貌和专业，并感谢那些帮助过你的人。,NaN,NaN,NaN
8,人工智能有哪些共同的挑战？,在应用人工智能解决不同行业问题时，会遇到一些共同的挑战。首先，需要明确的是要识别一个业务问题，而不是单纯的人工智能技术问题。这意味着你需要找到领域内的专家，并了解他们希望哪些方面能够得到改进以及为什么目前还没有实现这些改进。其次，在确定了具体的业务问题后，还需要思考如何利用人工智能技术来提出解决方案。此外，跨领域的合作和沟通也是一个挑战，因为非该领域的专家可能对该行业的具体细节不够熟悉。最后，建立和扩展专业网络对于寻找合适的项目和机会也非常重要，可以通过参加相关的聚会活动或与已经在该领域工作的人进行信息性访谈来实现这一点。,1.0,0.0,NaN
9,发现AI的某些部分具有挑战性是正常的吗？,是的，发现AI的某些部分具有挑战性是很正常的。在尝试解决业务问题时，理解问题本身、找到合适的解决方案以及实施这些方案都可能遇到各种挑战。与领域专家合作，了解他们最希望改进的方面，并探讨潜在的AI解决方案，可以帮助更有效地应对这些挑战。,1.0,0.0,0.833333


In [43]:
tru.get_leaderboard(app_ids=[])

,,Answer Relevance,Context Relevance,Groundedness,latency,total_cost
app_name,app_version,,,,,
App_1,base,0.833333,0.222222,0.786667,16.491283,0.0


In [69]:
tru.run_dashboard()

Starting dashboard ...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:53306 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

In [70]:
tru.stop_dashboard()